In [28]:
import flowArcFirstTry as eh
import json
import pandas as pd
import numpy as np
from copy import deepcopy

In [29]:
file_path = '../new_test_instances.json'
df = pd.read_json(file_path)

# Auswahl und Anordnung der Spalten mit den Nullen an den exakten Positionen
instance_patients = np.array([
    [
        row['patient_id'],
        row['arrival_time'],
        row['realized_processing_time'],
        0, 0,row['arrival_time']+row['max_wait_time'], 0, 
        row['weight'],
        row['expected_processing_time']
    ]
    for _, row in df.iterrows()
], dtype=int)

# Ergebnis anzeigen
print(instance_patients)

[[  1 486  12   0   0 501   0   3  14]
 [  2 486  22   0   0 546   0   2  20]
 [  3 486  19   0   0 606   0   1  19]
 [  4 488  15   0   0 548   0   2  15]
 [  5 495  28   0   0 555   0   2   5]
 [  6 497  23   0   0 557   0   2  20]
 [  7 499  19   0   0 559   0   2  13]
 [  8 500  16   0   0 560   0   2   1]
 [  9 507   9   0   0 522   0   3  30]
 [ 10 513  13   0   0 573   0   2  19]
 [ 11 518   9   0   0 533   0   3  28]
 [ 12 524  15   0   0 584   0   2   6]
 [ 13 526  13   0   0 586   0   2  23]
 [ 14 527  24   0   0 587   0   2  18]
 [ 15 535  24   0   0 595   0   2  21]
 [ 16 540  17   0   0 555   0   3  17]
 [ 17 540  11   0   0 600   0   2  18]
 [ 18 540  15   0   0 600   0   2  24]
 [ 19 540  14   0   0 555   0   3  15]
 [ 20 540  16   0   0 600   0   2  17]
 [ 21 540  20   0   0 600   0   2  14]
 [ 22 549  10   0   0 609   0   2  22]
 [ 23 556  19   0   0 571   0   3  40]
 [ 24 557  24   0   0 617   0   2  19]
 [ 25 557  11   0   0 617   0   2  20]
 [ 26 557  21   0   0 572

In [30]:
instance_patients

array([[  1, 486,  12,   0,   0, 501,   0,   3,  14],
       [  2, 486,  22,   0,   0, 546,   0,   2,  20],
       [  3, 486,  19,   0,   0, 606,   0,   1,  19],
       [  4, 488,  15,   0,   0, 548,   0,   2,  15],
       [  5, 495,  28,   0,   0, 555,   0,   2,   5],
       [  6, 497,  23,   0,   0, 557,   0,   2,  20],
       [  7, 499,  19,   0,   0, 559,   0,   2,  13],
       [  8, 500,  16,   0,   0, 560,   0,   2,   1],
       [  9, 507,   9,   0,   0, 522,   0,   3,  30],
       [ 10, 513,  13,   0,   0, 573,   0,   2,  19],
       [ 11, 518,   9,   0,   0, 533,   0,   3,  28],
       [ 12, 524,  15,   0,   0, 584,   0,   2,   6],
       [ 13, 526,  13,   0,   0, 586,   0,   2,  23],
       [ 14, 527,  24,   0,   0, 587,   0,   2,  18],
       [ 15, 535,  24,   0,   0, 595,   0,   2,  21],
       [ 16, 540,  17,   0,   0, 555,   0,   3,  17],
       [ 17, 540,  11,   0,   0, 600,   0,   2,  18],
       [ 18, 540,  15,   0,   0, 600,   0,   2,  24],
       [ 19, 540,  14,   0, 

In [31]:
import Neighbourhoods as nh

def add_patient(waiting_room_schedule, doctor, patient):
    waiting_room_schedule[doctor] = np.vstack([waiting_room_schedule[doctor], patient])

def call_next_patient(waiting_room_schedule, doctor):
    if waiting_room_schedule[doctor].shape[0] > 0:  # Überprüfen, ob das Array nicht leer ist
        patient = waiting_room_schedule[doctor][0]
        delete_patient_form_waiting_room(waiting_room_schedule, doctor)
        return patient
    else:
        return np.array([])

def delete_patient_form_waiting_room(waiting_room_schedule, doctor):
    waiting_room_schedule[doctor] = waiting_room_schedule[doctor][1:]

def start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end , doctor):
    patient = call_next_patient(waiting_room_schedule, doctor)
    if patient.shape[0]>0:
        estimated_treatment_end[doctor-1] = time + patient[8]
        treatment_end[doctor-1] = time+patient[2]
        #print(f"{time}: Beginn Treatment for {patient}")
        schedule_treated_patient(schedule, doctor, patient)
        nh.calculate_start_and_endtimes_per_doc(schedule, doctor)

def calculate_completion_time_per_doc(waiting_room_schedule, doctor_completion, estimated_treatment_end, doctor, deterministic=True):
    calculate_waiting_room_values_per_doc(waiting_room_schedule, doctor, estimated_treatment_end, deterministic=deterministic)
    doctor_completion[doctor-1] = waiting_room_schedule[doctor][-1, 4]

def schedule_treated_patient(schedule, doctor, patient):
    add_patient(schedule, doctor, patient)

def greedy_heuristic_per_patient(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient, deterministic=True):
    next_available_doctor = np.argmin(doctor_completion) + 1
    doctor_completion[next_available_doctor-1] += patient[8]
    add_patient(waiting_room_schedule, next_available_doctor, patient)
    reprioritize_patients_by_urgency_level(waiting_room_schedule, next_available_doctor)
    estimated_treatment_end_doctor = estimated_treatment_end[next_available_doctor-1]
    calculate_completion_time_per_doc(waiting_room_schedule, doctor_completion, estimated_treatment_end_doctor, next_available_doctor, deterministic=deterministic)
    
def reprioritize_patients_by_urgency_level(waiting_room_schedule, doctor):
    sorted_patients_indices = np.lexsort((
    waiting_room_schedule[doctor][:, 1],  # 2. Spalte
    -waiting_room_schedule[doctor][:, 7]   # 8. Spalte
))
    #sorted_patients_indices = np.argsort(waiting_room_schedule[doctor][:, 7])[::-1]
    waiting_room_schedule[doctor] = waiting_room_schedule[doctor][sorted_patients_indices]

def dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient_list, deterministic=True):
    incoming_patient_priority_by_weights_indices = np.argsort(patient_list[:, 7])[::-1]
    incoming_patient_priority_by_weights = patient_list[incoming_patient_priority_by_weights_indices]

    for patient in incoming_patient_priority_by_weights:
        #print(f"{patient}")
        greedy_heuristic_per_patient(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient, deterministic=deterministic)

    # for patient in patient_list:
    #     greedy_heuristic_per_patient(waiting_room_schedule, doctor_completion, estimated_treatment_end, patient)
        
    
# def calculate_waiting_room_values_per_doc(waiting_room_schedule, doctor, estimated_treatment_end):
#     for i in range(len(waiting_room_schedule[doctor])):
#         if i == 0:
#             waiting_room_schedule[doctor][0, [4]] = estimated_treatment_end + waiting_room_schedule[doctor][0, [8]]
#             waiting_room_schedule[doctor][0, [3]] = estimated_treatment_end
#         else:
#             waiting_room_schedule[doctor][i, [3]] = max(waiting_room_schedule[doctor][i-1, [4]], waiting_room_schedule[doctor][i, [1]])
#             waiting_room_schedule[doctor][i, [4]] = waiting_room_schedule[doctor][i, [3]] + waiting_room_schedule[doctor][i, [8]]
    
#     # Berechne das Maximum aus 0 und der Differenz der 4. und 6. Spalte
#     result = np.maximum(0, waiting_room_schedule[doctor][:, 3] - waiting_room_schedule[doctor][:, 5])

#     # Weisen Sie das Ergebnis der 7. Spalte zu
#     waiting_room_schedule[doctor][:, 6] = result    

# def calculate_waiting_room_values_per_doc(schedule, key, start_time=0):
#     for i in range(len(schedule[key])):
#         if i == 0 and start_time == 0:
#             schedule[key][0, [4]] = schedule[key][0, [1]] + schedule[key][0, [2]]
#             schedule[key][0, [3]] = schedule[key][0, [1]]
#         elif i == 0 and start_time > 0:
#             schedule[key][0, [4]] = start_time + schedule[key][0, [8]]
#             schedule[key][0, [3]] = start_time
#         else:
#             schedule[key][i, [3]] = max(schedule[key][i-1, [4]], schedule[key][i, [1]])
#             schedule[key][i, [4]] = schedule[key][i, [3]] + schedule[key][i, [2]]

def calculate_waiting_room_values_per_doc(schedule, key, start_time=0,deterministic=True):
    if deterministic:
        nh.calculate_start_and_endtimes_per_doc(schedule, key, start_time)
    else:
        nh.calculate_start_and_endtimes_per_doc_non_det(schedule, key, start_time)
    

In [57]:
# Waiting_room
doctor_count = 1
doctor_completion = np.zeros(doctor_count)

waiting_room_schedule = {}
waiting_room_weighted_tardiness = 0
schedule = {}
deterministic = False
metaheuristic = "VNS"
#metaheuristic = 0
#metaheuristic = "SA"

for i in range(1, doctor_count+1):
    waiting_room_schedule[i] = np.array([], dtype=int).reshape(0, 9)

schedule = deepcopy(waiting_room_schedule)

start_time = instance_patients[0][1]
end_time = instance_patients[-1][1]

#start_time = 486
#end_time = 550

estimated_treatment_end = np.full(doctor_count, start_time)
treatment_end = estimated_treatment_end.copy()
#new_patient = False
time = start_time

while time <= end_time:
    arrived_patients = instance_patients[:, 1] == time

    arrived_patients_list = instance_patients[arrived_patients]
    if arrived_patients_list.shape[0] > 0:
        #new_patient = True
        #print(f"{time}: Patient arrives:")
        dynamic_greedy_heuristic(waiting_room_schedule, doctor_completion, estimated_treatment_end, arrived_patients_list, deterministic=deterministic)
        
        for doctor in waiting_room_schedule:
            waiting_room_weighted_tardiness = 0
            waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        

        # Metaheuristik
        if metaheuristic == "VNS" and waiting_room_weighted_tardiness != 0:
            new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.general_vns(waiting_room_schedule, waiting_room_weighted_tardiness, ["N1", "N2", "N3"], Max_d=120, no_improvement_limit=10, time_limit=600, start_time_array=estimated_treatment_end, deterministic=deterministic)
            print(f"{time}: VNS {waiting_room_weighted_tardiness} auf {new_waiting_room_weighted_tardiness}")
            waiting_room_schedule = deepcopy(new_waiting_room_schedule)
            waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
        
        #new_patient = False, 
        # waiting_room_weighted_tardiness = 0
        # for doctor in waiting_room_schedule:
        #     waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)
        #print(f"Time {time} waiting room tardiness = {waiting_room_weighted_tardiness}")
    
    #feasible_metaheuristic = any(array.size > 1 for array in waiting_room_schedule.values())

        

    free_doctors = np.where(treatment_end <= time)[0]
    free_doctors += 1

    if len(free_doctors)>0:
        for doctor in free_doctors:
            start_patient_treatment(waiting_room_schedule, schedule, time, estimated_treatment_end, treatment_end, doctor)
            
            #print(f"estimated_end: {estimated_treatment_end}, treatment_end: {treatment_end}, completion_time: {doctor_completion}")
            #for doctor in waiting_room_schedule:
                #print(f"Waiting_room_schedule: {waiting_room_schedule[doctor]}")
            #print(f"Schedule:")
            #print(schedule)
            if metaheuristic == "SA" and waiting_room_weighted_tardiness != 0:
                #print(f"{time} SA triggered with weighted tardiness: {waiting_room_weighted_tardiness}\n{waiting_room_schedule}")
                new_waiting_room_schedule, new_waiting_room_weighted_tardiness, _ = nh.simulated_annealing(waiting_room_schedule, waiting_room_weighted_tardiness, start_time_array=estimated_treatment_end, start_temperature=10, Imax=2000, time_limit=10, Max_d=1200, deterministic=deterministic)
                print(f"SA Done")
                if waiting_room_weighted_tardiness > new_waiting_room_weighted_tardiness:
                    print(f"{time}: SA {waiting_room_weighted_tardiness} auf {new_waiting_room_weighted_tardiness}")
                    waiting_room_schedule = deepcopy(new_waiting_room_schedule)
                    #waiting_room_weighted_tardiness = deepcopy(new_waiting_room_weighted_tardiness)
        for doctor in waiting_room_schedule:
            waiting_room_weighted_tardiness = 0
            waiting_room_weighted_tardiness += nh.weighted_tardiness_per_doc(waiting_room_schedule, doctor)

    #print(f"{time}: completion_time: {doctor_completion}, estimated_end: {estimated_treatment_end} and end: {treatment_end}")
    #for doctor in waiting_room_schedule:
        #print(f"{doctor}:\n {waiting_room_schedule[doctor]}")
        #print(f"{doctor} schedule :\n {schedule[doctor]}")
    time += 1
    waiting_room_empy = all(array.size == 0 for array in waiting_room_schedule.values())
    if not waiting_room_empy:
        end_time += 1
    
    # if arrived_patients_list.shape[0] > 0:
    #     #print(f" Patient: {arrived_patients_list} \n estimated_end: {estimated_treatment_end}, end: {treatment_end}; completion_time: {doctor_completion}")
    #     for doctor in waiting_room_schedule:
    #        #print(f"doctor {doctor} waiting_room: \n {waiting_room_schedule[doctor]} \n")
    #        #print(f"doctor {doctor} schedule: \n {schedule[doctor]} \n")

schedule_weighted_tardiness = 0
for doctor in schedule:
    schedule_weighted_tardiness += nh.weighted_tardiness_per_doc(schedule, doctor)

print(f"{schedule}, {schedule_weighted_tardiness}")

Bessere Lösung 0.0: 22 auf 0
No improvement for 10 iterations
500: VNS 22 auf 0
Bessere Lösung 0.0: 178 auf 102
Bessere Lösung 0.01620006561279297: 102 auf 82
No improvement for 10 iterations
507: VNS 178 auf 82
Bessere Lösung 0.016988754272460938: 251 auf 207
Bessere Lösung 0.03606581687927246: 207 auf 200
Bessere Lösung 0.06508660316467285: 200 auf 155
Bessere Lösung 0.13738441467285156: 155 auf 153
No improvement for 10 iterations
513: VNS 251 auf 153
Bessere Lösung 0.0: 660 auf 468
Bessere Lösung 0.023116111755371094: 468 auf 448
Bessere Lösung 0.06111741065979004: 448 auf 440
No improvement for 10 iterations
518: VNS 660 auf 440
Bessere Lösung 0.0: 832 auf 752
Bessere Lösung 0.02299785614013672: 752 auf 629
Bessere Lösung 0.04299783706665039: 629 auf 580
Bessere Lösung 0.0640709400177002: 580 auf 544
Bessere Lösung 0.10807013511657715: 544 auf 537
Bessere Lösung 0.1693408489227295: 537 auf 533
Bessere Lösung 0.20898079872131348: 533 auf 512
No improvement for 10 iterations
524: VN

In [33]:
waiting_room_schedule

{1: array([], shape=(0, 9), dtype=int32),
 2: array([], shape=(0, 9), dtype=int32)}

In [34]:
schedule

{1: array([[  1, 486,  12, 486, 498, 501,   0,   3,  14],
        [  5, 495,  28, 498, 526, 555,   0,   2,   5],
        [  9, 507,   9, 526, 535, 522,   4,   3,  30],
        [  6, 497,  23, 535, 558, 557,   0,   2,  20],
        [ 16, 540,  17, 558, 575, 555,   3,   3,  17],
        [ 23, 556,  19, 575, 594, 571,   4,   3,  40],
        [ 27, 557,  24, 594, 618, 572,  22,   3,  26],
        [ 30, 569,  18, 618, 636, 584,  34,   3,  16],
        [ 31, 578,  10, 636, 646, 593,  43,   3,  29],
        [ 34, 603,  11, 646, 657, 618,  28,   3,  15],
        [ 38, 628,  10, 657, 667, 643,  14,   3,  15],
        [ 42, 642,  25, 667, 692, 657,  10,   3,  31],
        [ 45, 661,  20, 692, 712, 676,  16,   3,  13],
        [ 49, 678,  18, 712, 730, 693,  19,   3,  23],
        [ 53, 713,  10, 730, 740, 728,   2,   3,  26],
        [ 12, 524,  15, 740, 755, 584, 156,   2,   6],
        [ 13, 526,  13, 755, 768, 586, 169,   2,  23],
        [ 15, 535,  24, 768, 792, 595, 173,   2,  21],
       

In [35]:
sched_det = deepcopy(schedule)
sched_non_det = deepcopy(schedule)


In [36]:
def calculate_waiting_room_values_per_doc(schedule, key, start_time=0, deterministic=True):
    for i in range(len(schedule[key])):
        if i == 0 and start_time == 0:
            schedule[key][0, [4]] = schedule[key][0, [1]] + schedule[key][0, [2]]
            schedule[key][0, [3]] = schedule[key][0, [1]]
        elif i == 0 and start_time > 0:
            schedule[key][0, [4]] = start_time + schedule[key][0, [8]]
            schedule[key][0, [3]] = start_time
        elif deterministic:
            schedule[key][i, [3]] = max(schedule[key][i-1, [4]], schedule[key][i, [1]])
            schedule[key][i, [4]] = schedule[key][i, [3]] + schedule[key][i, [2]]
        else:
            schedule[key][i, [3]] = max(schedule[key][i-1, [4]], schedule[key][i, [1]])
            schedule[key][i, [4]] = schedule[key][i, [3]] + schedule[key][i, [8]]

    # Berechne das Maximum aus 0 und der Differenz der 4. und 6. Spalte
    result = np.maximum(0, schedule[key][:, 3] - schedule[key][:, 5])

    # Weisen Sie das Ergebnis der 7. Spalte zu
    schedule[key][:, 6] = result    

In [37]:
calculate_waiting_room_values_per_doc(sched_det, 1)
calculate_waiting_room_values_per_doc(sched_non_det, 1, deterministic=False)

In [38]:
schedule

{1: array([[  1, 486,  12, 486, 498, 501,   0,   3,  14],
        [  5, 495,  28, 498, 526, 555,   0,   2,   5],
        [  9, 507,   9, 526, 535, 522,   4,   3,  30],
        [  6, 497,  23, 535, 558, 557,   0,   2,  20],
        [ 16, 540,  17, 558, 575, 555,   3,   3,  17],
        [ 23, 556,  19, 575, 594, 571,   4,   3,  40],
        [ 27, 557,  24, 594, 618, 572,  22,   3,  26],
        [ 30, 569,  18, 618, 636, 584,  34,   3,  16],
        [ 31, 578,  10, 636, 646, 593,  43,   3,  29],
        [ 34, 603,  11, 646, 657, 618,  28,   3,  15],
        [ 38, 628,  10, 657, 667, 643,  14,   3,  15],
        [ 42, 642,  25, 667, 692, 657,  10,   3,  31],
        [ 45, 661,  20, 692, 712, 676,  16,   3,  13],
        [ 49, 678,  18, 712, 730, 693,  19,   3,  23],
        [ 53, 713,  10, 730, 740, 728,   2,   3,  26],
        [ 12, 524,  15, 740, 755, 584, 156,   2,   6],
        [ 13, 526,  13, 755, 768, 586, 169,   2,  23],
        [ 15, 535,  24, 768, 792, 595, 173,   2,  21],
       

In [46]:
nh.weighted_tardiness_per_doc(sched_det, 1) + nh.weighted_tardiness_per_doc(sched_det, 2)

9560

In [44]:
schedule

{1: array([[  1, 486,  12, 486, 498, 501,   0,   3,  14],
        [  5, 495,  28, 498, 526, 555,   0,   2,   5],
        [  9, 507,   9, 526, 535, 522,   4,   3,  30],
        [  6, 497,  23, 535, 558, 557,   0,   2,  20],
        [ 16, 540,  17, 558, 575, 555,   3,   3,  17],
        [ 23, 556,  19, 575, 594, 571,   4,   3,  40],
        [ 27, 557,  24, 594, 618, 572,  22,   3,  26],
        [ 30, 569,  18, 618, 636, 584,  34,   3,  16],
        [ 31, 578,  10, 636, 646, 593,  43,   3,  29],
        [ 34, 603,  11, 646, 657, 618,  28,   3,  15],
        [ 38, 628,  10, 657, 667, 643,  14,   3,  15],
        [ 42, 642,  25, 667, 692, 657,  10,   3,  31],
        [ 45, 661,  20, 692, 712, 676,  16,   3,  13],
        [ 49, 678,  18, 712, 730, 693,  19,   3,  23],
        [ 53, 713,  10, 730, 740, 728,   2,   3,  26],
        [ 12, 524,  15, 740, 755, 584, 156,   2,   6],
        [ 13, 526,  13, 755, 768, 586, 169,   2,  23],
        [ 15, 535,  24, 768, 792, 595, 173,   2,  21],
       

In [45]:
sched_det

{1: array([[  1, 486,  12, 486, 498, 501,   0,   3,  14],
        [  5, 495,  28, 498, 526, 555,   0,   2,   5],
        [  9, 507,   9, 526, 535, 522,   4,   3,  30],
        [  6, 497,  23, 535, 558, 557,   0,   2,  20],
        [ 16, 540,  17, 558, 575, 555,   3,   3,  17],
        [ 23, 556,  19, 575, 594, 571,   4,   3,  40],
        [ 27, 557,  24, 594, 618, 572,  22,   3,  26],
        [ 30, 569,  18, 618, 636, 584,  34,   3,  16],
        [ 31, 578,  10, 636, 646, 593,  43,   3,  29],
        [ 34, 603,  11, 646, 657, 618,  28,   3,  15],
        [ 38, 628,  10, 657, 667, 643,  14,   3,  15],
        [ 42, 642,  25, 667, 692, 657,  10,   3,  31],
        [ 45, 661,  20, 692, 712, 676,  16,   3,  13],
        [ 49, 678,  18, 712, 730, 693,  19,   3,  23],
        [ 53, 713,  10, 730, 740, 728,   2,   3,  26],
        [ 12, 524,  15, 740, 755, 584, 156,   2,   6],
        [ 13, 526,  13, 755, 768, 586, 169,   2,  23],
        [ 15, 535,  24, 768, 792, 595, 173,   2,  21],
       

In [47]:
nh.weighted_tardiness_per_doc(sched_non_det, 1) + nh.weighted_tardiness_per_doc(sched_non_det, 2)

12197

In [41]:
schedule_weighted_tardiness

9560

In [ ]:
#nh.general_vns(initial_schedule=schedule, initial_tardiness=schedule_weighted_tardiness, neighborhoods=["N1", "N2", "N3"], Max_d=120, start_time_array=[0, 0],no_improvement_limit=10, time_limit=600)

Bessere Lösung 3.098450183868408: 9560 auf 9349
Bessere Lösung 6.124870777130127: 9349 auf 9233
Bessere Lösung 8.123159885406494: 9233 auf 9079
Bessere Lösung 11.187980890274048: 9079 auf 8940


KeyboardInterrupt: 

In [ ]:
nh.weighted_tardiness_per_doc(schedule, 1)

6840

In [ ]:
waiting_room_schedule

{1: array([], shape=(0, 9), dtype=int32),
 2: array([], shape=(0, 9), dtype=int32)}